# IT1244 Group Project: CGM Device Failure Classification

This notebook runs the whole project in order: load the data, split it, analyse it, clean it, train the models and evaluate them. The code it calls is in `src/` (see README.md).

**Team:** Lim Su Ling, Ho Zi Yun Leanne, James Elliot Toh Zhiheng, Chng Lucas

**Task:** multi-class classification. Predict each report's `primary_category` (its IMDRF problem code, e.g. A07 Electrical, A09 Output, A13 Communication) from the free-text `event_description` of FDA MAUDE adverse event reports for continuous glucose monitors (CGMs).

**Model input:** `event_description` only. `brand_name`, `problem_terms`, `imdrf_top_categories` and `manufacturer_narrative` are used for analysis, never as model input.

| Section | Owner |
|---|---|
| 2. Train/test split | Lucas |
| 3. Dataset analysis | Leanne |
| 1. Loading, 4. Data cleaning | Suling, James |
| 5.1 TF-IDF + LR | James |
| 5.2 BiLSTM, 5.3 transformer | Suling |
| 6. Evaluation (incl. majority baseline) | Lucas |

## References and how we address their limitations

1. **Wang, Kang & Gong (2019), MEDINFO 2019.** *Generating a Health Information Technology Event Database from FDA MAUDE Reports.* TF-IDF models (LR, SVM, NB, RF) vs a CNN on 5,588 MAUDE reports; LR alone reached F1 0.859, the best ensemble 0.870.
   *Limitations and our approach:* their models ignore word context, the data contained duplicates, and it was artificially balanced. We use TF-IDF + LR only as our baseline and test whether models that read word order (BiLSTM) and context (transformer) do better; we keep duplicate descriptions on one side of the split; and we keep the real class imbalance, handling it with class weights and macro-F1.
2. **Li et al. (2024), Frontiers in Computational Neuroscience.** BiLSTM with attention for harm vs no-harm classification of 4,568 da Vinci surgical-robot MAUDE reports; F1 85.37, above plain BERT (78.61).
   *Limitations and our approach:* a small, single-device, binary dataset whose authors note performance on larger data is unvalidated. We test a BiLSTM (with attention) on ~380k CGM reports with a multi-class label, using dropout and early stopping to limit overfitting.
3. **Luschi, Nesi & Iadanza (2023), Heliyon.** Fine-tuned ClinicalBERT on 3,075 expert-labelled MAUDE reports to detect health-IT adverse events, with explainability.
   *Limitations and our approach:* a small dataset with no comparison against cheaper baselines. We fine-tune DistilBERT / Bio_ClinicalBERT on CGM reports and compare it directly with TF-IDF + LR and the BiLSTM on the same split and metrics, to see whether the extra compute is worth it.

Background: CGM failure modes in MAUDE, https://pmc.ncbi.nlm.nih.gov/articles/PMC11688675/

TODO: add the full links for references 1-3 from the proposal doc.

**Code attribution:** the transformer uses the Hugging Face `transformers` library and pretrained weights (`distilbert-base-uncased`, `emilyalsentzer/Bio_ClinicalBERT`).

## 0. Setup

1. Put the course CGM MAUDE file in `data/` (see `data/README.md`).
2. To **reproduce our results without training**, download the weights (see `links.txt`) into `weights/` and set `TRAIN = False`.
3. Run all cells top to bottom.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src import config
from src.analysis import eda
from src.data import cleaning, loading, splitting
from src.evaluation import cross_validation, metrics
from src.evaluation.baseline import MajorityBaseline
from src.helpers import device_info, get_device, set_seed
from src.models.bilstm import BiLSTMClassifier
from src.models.tfidf_lr import TfidfLR
from src.models.transformer import CLINICAL_BERT, DISTILBERT, TransformerClassifier

TRAIN = True   # False: skip training and load the saved weights from weights/
RUN_CV = True  # 5-fold cross-validation to choose hyper-parameters (slow for the BiLSTM)

set_seed(config.SEED)
device = get_device()
config.WEIGHTS_DIR.mkdir(exist_ok=True)
print(device_info())

## 1. Load data (Suling, James)

In [ ]:
df = loading.load_reports()
print(df.shape)
df.head()

## 2. Train/test split (Lucas)

Stratified 80/20 split on `primary_category` with a fixed seed, with identical (template) descriptions kept on one side so test scores are not inflated. The test set is used **once**, in section 6.

In [ ]:
train_df, test_df = splitting.split_train_test(df)
print(f"train {len(train_df):,} | test {len(test_df):,}")

## 3. Dataset analysis on the training set (Leanne)

### 3.1 Class counts
Informs: class weights, which rare classes to merge, and why macro-F1 is the main metric.

In [ ]:
eda.plot_class_counts(train_df[config.LABEL_COL], save_as="class_counts.png")
eda.class_counts(train_df[config.LABEL_COL])

**Findings:** TODO (Leanne)

### 3.2 Text length
Informs: the maximum input length for the BiLSTM and transformer.

In [ ]:
display(eda.length_percentiles(train_df[config.TEXT_COL]))
eda.plot_length_hist(train_df[config.TEXT_COL], save_as="length_hist.png")
eda.plot_length_by_class(train_df, save_as="length_by_class.png")

**Findings:** TODO (Leanne)

### 3.3 Empty or very short descriptions
Informs: whether cleaning drops these reports or keeps them.

In [ ]:
eda.short_description_summary(train_df[config.TEXT_COL])

**Findings:** TODO (Leanne)

### 3.4 Duplicate (template) descriptions

In [ ]:
duplicate_stats, top_templates = eda.duplicate_summary(train_df)
display(duplicate_stats, top_templates)

**Findings:** TODO (Leanne)

### 3.5 Reports by brand
Checks whether failure types and report style (templated vs free text) differ by brand.

In [ ]:
display(eda.brand_summary(train_df))
eda.plot_brand_by_class(train_df, save_as="brand_by_class.png")

**Findings:** TODO (Leanne)

### 3.6 Reports with more than one IMDRF category
Informs: evaluating single- and multi-category reports separately (section 6.4).

In [ ]:
eda.plot_imdrf_code_counts(train_df[config.IMDRF_COL], save_as="imdrf_codes.png")

**Findings:** TODO (Leanne)

### 3.7 Most frequent words and phrases per category
Informs: text-cleaning steps and the TF-IDF n-gram range.

In [ ]:
eda.top_ngrams_per_class(train_df[config.TEXT_COL], train_df[config.LABEL_COL], k=15)

**Findings:** TODO (Leanne)

## 4. Data cleaning (Suling, James)

Steps chosen from the analysis in section 3 (see `src/data/cleaning.py`). The same steps are applied to the training and test sets; rare classes are found from the training set only.

In [ ]:
train_df = cleaning.clean_reports(train_df)
test_df = cleaning.clean_reports(test_df)

rare_classes = cleaning.find_rare_classes(train_df[config.LABEL_COL])
train_df[config.LABEL_COL] = cleaning.merge_classes(train_df[config.LABEL_COL], rare_classes)
test_df[config.LABEL_COL] = cleaning.merge_classes(test_df[config.LABEL_COL], rare_classes)

X_train, y_train = train_df[config.TEXT_COL], train_df[config.LABEL_COL]
X_test, y_test = test_df[config.TEXT_COL], test_df[config.LABEL_COL]

# Cross-validation folds on the cleaned training set, shared by all models
folds = splitting.cv_folds(train_df)

# Max input length in words for the BiLSTM and transformer: long enough for 95% of reports
MAX_LEN = int(eda.length_percentiles(X_train)["p95"])
print("MAX_LEN =", MAX_LEN)

## 5. Models

Every model has the same methods (`src/interfaces.py`): `fit(texts, labels)`, `predict(texts)`, `predict_proba(texts)`, `save(path)` and `load(path)`. All use class weighting for the imbalance and the same split and folds.

### 5.1 TF-IDF + Logistic Regression (James)

Word/phrase counts weighted by TF-IDF, then logistic regression with balanced class weights. Cross-validated over the n-gram range and regularisation strength `C`.

In [ ]:
tfidf_path = config.WEIGHTS_DIR / "tfidf_lr.joblib"
tfidf_params = dict(ngram_range=(1, 2), C=1.0)  # used if cross-validation is skipped

if TRAIN and RUN_CV:
    tfidf_grid = [dict(ngram_range=(1, 1), C=1.0),
                  dict(ngram_range=(1, 2), C=1.0),
                  dict(ngram_range=(1, 2), C=10.0)]
    tfidf_cv, tfidf_params = cross_validation.grid_search(TfidfLR, tfidf_grid, X_train, y_train, folds)
    display(tfidf_cv)

if TRAIN:
    tfidf = TfidfLR(**tfidf_params).fit(X_train, y_train)
    tfidf.save(tfidf_path)
else:
    tfidf = TfidfLR.load(tfidf_path)

tfidf.top_features(k=10)  # compare with the phrases in section 3.7

### 5.2 Bidirectional LSTM (Suling)

Word embeddings → BiLSTM (reads the report in both directions) → attention → linear layer. Class-weighted loss, dropout, and early stopping on 10% of the training data (never the test set). Cross-validated over hidden size and dropout.

In [ ]:
bilstm_path = config.WEIGHTS_DIR / "bilstm.pt"

def make_bilstm(**params):
    return BiLSTMClassifier(max_len=MAX_LEN, epochs=10, patience=2, device=device, **params)

bilstm_params = dict(hidden_dim=128, dropout=0.3)  # used if cross-validation is skipped

if TRAIN and RUN_CV:
    bilstm_grid = [dict(hidden_dim=128, dropout=0.3),
                   dict(hidden_dim=256, dropout=0.3),
                   dict(hidden_dim=128, dropout=0.5)]
    bilstm_cv, bilstm_params = cross_validation.grid_search(make_bilstm, bilstm_grid, X_train, y_train, folds)
    display(bilstm_cv)

if TRAIN:
    bilstm = make_bilstm(**bilstm_params).fit(X_train, y_train)
    bilstm.save(bilstm_path)
    display(pd.DataFrame(bilstm.history_))
else:
    bilstm = BiLSTMClassifier.load(bilstm_path, device=device)

### 5.3 Fine-tuned transformer (Suling)

A pretrained transformer with a new classification layer, fine-tuned on our reports. It needs a GPU (e.g. Colab); otherwise train on a smaller stratified sample. Standard fine-tuning settings (learning rate 2e-5, up to 3 epochs, early stopping) instead of cross-validation, because each run is expensive.

In [ ]:
transformer_path = config.WEIGHTS_DIR / "transformer"
TRANSFORMER_MODEL = DISTILBERT       # or CLINICAL_BERT
TRANSFORMER_TRAIN_SAMPLES = 50_000   # stratified sample to save compute; None = all training data

if TRAIN:
    sample = train_df
    if TRANSFORMER_TRAIN_SAMPLES and TRANSFORMER_TRAIN_SAMPLES < len(train_df):
        sample, _ = train_test_split(train_df, train_size=TRANSFORMER_TRAIN_SAMPLES,
                                     stratify=train_df[config.LABEL_COL], random_state=config.SEED)

    transformer = TransformerClassifier(
        model_name=TRANSFORMER_MODEL,
        max_len=min(512, int(MAX_LEN * 1.3)),  # a word is ~1.3 sub-word tokens; 512 is the model limit
        device=device,
    ).fit(sample[config.TEXT_COL], sample[config.LABEL_COL])
    transformer.save(transformer_path)
    display(pd.DataFrame(transformer.history_))
else:
    transformer = TransformerClassifier.load(transformer_path, device=device)

## 6. Evaluation on the test set (Lucas)

The test set is used once, here. Main metric is **macro-F1** (every category counts equally, so a model cannot score well by ignoring rare failure types).

### 6.1 Overall comparison

The first row is the majority baseline (`src/evaluation/baseline.py`), which always predicts the most common training category. It is the minimum bar every model must beat, and its high accuracy but very low macro-F1 shows why macro-F1 is the main metric.

In [ ]:
baseline = MajorityBaseline().fit(X_train, y_train)

models = {"Majority baseline": baseline, "TF-IDF + LR": tfidf, "BiLSTM": bilstm,
          "Transformer": transformer}
predictions = {name: model.predict(X_test) for name, model in models.items()}

pd.DataFrame({name: metrics.scores(y_test, pred) for name, pred in predictions.items()}).T

### 6.2 Per-class results and confusion matrices

In [ ]:
for name, file_name in [("TF-IDF + LR", "confusion_tfidf.png"),
                        ("BiLSTM", "confusion_bilstm.png"),
                        ("Transformer", "confusion_transformer.png")]:
    print(name)
    display(metrics.per_class_report(y_test, predictions[name]))
    metrics.plot_confusion_matrix(y_test, predictions[name], title=name, save_as=file_name)

### 6.3 Macro-F1 by brand

In [ ]:
pd.DataFrame({name: metrics.macro_f1_by_group(y_test, pred, test_df[config.BRAND_COL])
              for name, pred in predictions.items()})

### 6.4 Macro-F1 on single- vs multi-category reports

In [ ]:
is_multi = eda.count_imdrf_codes(test_df[config.IMDRF_COL]) > 1
report_type = np.where(is_multi, "multi-category", "single-category")
pd.DataFrame({name: metrics.macro_f1_by_group(y_test, pred, report_type)
              for name, pred in predictions.items()})

### 6.5 Binary extension: defect vs not

Recall comes first: missing a real defect costs more than a false alarm. A model's defect probability is its total probability over the defect categories.

In [ ]:
DEFECT_CLASSES = []  # TODO (Lucas): the primary_category codes that count as a defect

if DEFECT_CLASSES:
    is_defect = y_test.isin(DEFECT_CLASSES).to_numpy().astype(int)
    binary_results = {}
    for name in ["TF-IDF + LR", "BiLSTM", "Transformer"]:
        model = models[name]
        defect_columns = np.isin(model.classes_, DEFECT_CLASSES)
        defect_probability = model.predict_proba(X_test)[:, defect_columns].sum(axis=1)
        binary_results[name] = metrics.binary_scores(is_defect, defect_probability)
    display(pd.DataFrame(binary_results).T)

### 6.6 Comparison with previous MAUDE studies

| Study | Data | Task | Best reported |
|---|---|---|---|
| Wang et al. 2019 | 5,588 MAUDE reports, balanced | binary (health-IT or not) | LR F1 0.859; SVM+LR+CNN F1 0.870 |
| Li et al. 2024 | 4,568 da Vinci reports | binary (harm or not) | BiLSTM-attention F1 85.37; BERT 78.61 |
| Luschi et al. 2023 | 3,075 labelled MAUDE reports | binary (health-IT or not) | fine-tuned ClinicalBERT |
| **Ours** | CGM MAUDE reports | multi-class IMDRF category | see 6.1 |

These studies used smaller datasets and binary labels, so the numbers are a rough reference, not a like-for-like comparison.

## 7. Discussion

TODO: which model is best and why (with evidence from 6.1-6.5), where models fail, comparison with human coders, and societal impact (privacy, fairness across brands, interpretability, jobs).